# Example 12.8.1 / 2: Train an L1 Support Vector Machine

Companion notebook for **12.8.1 Optimization Models in ML Pipelines** in the AMPL lab Documentation, section "Train an L1 Support Vector Machine". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Train an L1 Support Vector Machine

The LP below splits $w$ into `WPos - WNeg` and solves with HiGHS. Six points have two features;
the classes differ in `f1` only. The run solves for two values of `lambda` and reads $w$ with
`value()`, then switches to the L2 objective `SquaredHinge` with the `objective` command (sent
with `ampl.eval`, since amplpy has no method for it).

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set OBS;                                # training observations
set FEATURES;                           # input features
param x {OBS, FEATURES};                # feature value per observation
param label {OBS} in {-1, 1};           # class label, -1 or +1
param lambda >= 0;                      # L1 regularization weight

var WPos {FEATURES} >= 0;               # positive part of w
var WNeg {FEATURES} >= 0;               # negative part of w
var b;                                  # bias term
var Hinge {OBS} >= 0;                   # hinge loss per observation

minimize RegularizedHinge:              # L1 penalty plus mean hinge loss
    lambda * sum {j in FEATURES} (WPos[j] + WNeg[j])
  + sum {i in OBS} Hinge[i] / card(OBS);

subject to Margin {i in OBS}:           # hinge at least 1 minus the signed margin
    Hinge[i] >= 1 - label[i]
        * (sum {j in FEATURES} (WPos[j] - WNeg[j]) * x[i,j] + b);

minimize SquaredHinge:                  # L2 penalty plus mean hinge loss
    lambda * sum {j in FEATURES} (WPos[j] - WNeg[j])^2
  + sum {i in OBS} Hinge[i] / card(OBS);

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
points = {"p1": (2, 0, 1), "p2": (3, 2, 1), "p3": (2.5, 1, 1),
          "n1": (0, 1, -1), "n2": (1, 2, -1), "n3": (0.5, 0, -1)}
ampl.set["OBS"] = list(points)
ampl.set["FEATURES"] = ["f1", "f2"]
ampl.param["x"] = {(i, f): v[k] for i, v in points.items()
                   for k, f in enumerate(["f1", "f2"])}
ampl.param["label"] = {i: v[2] for i, v in points.items()}

In [5]:
# run.py
for lam in [0.1, 0.3]:
    ampl.param["lambda"] = lam
    ampl.solve(solver="highs", verbose=False)
    w = {j: round(ampl.var["WPos"][j].value() - ampl.var["WNeg"][j].value(), 4)
         for j in ["f1", "f2"]}
    print(lam, ampl.solve_result, w, round(ampl.var["b"].value(), 4),
          round(ampl.obj["RegularizedHinge"].value(), 4))

ampl.eval("objective SquaredHinge;")
ampl.solve(solver="highs", verbose=False)
w = {j: round(ampl.var["WPos"][j].value() - ampl.var["WNeg"][j].value(), 4)
     for j in ["f1", "f2"]}
print("L2", ampl.solve_result, w, round(ampl.var["b"].value(), 4))

0.1 solved {'f1': 1.3333, 'f2': -0.3333} -1.6667 0.1667
0.3 solved {'f1': 1.0, 'f2': 0.0} -1.5 0.4667
L2 solved {'f1': 0.8333, 'f2': -0.2778} -0.8056
